# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. All elements are referenced by `@id` as per Croissant specification.

In [ ]:
# List all available record sets in the dataset via their @id and display their schema fields

print("Available record sets and fields:")
record_sets = metadata.record_sets

if not record_sets:
    print("No record sets found in this dataset. Check the Croissant schema.")
else:
    for rs in record_sets:
        print(f"\nRecord Set @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        print("  Fields (@id : name):")
        for field in rs.fields:
            print(f"    {field.id} : {field.name} (type: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and fields are referenced by their `@id`.

For demonstration, we'll extract all available record sets into Pandas DataFrames for ease of exploration.

In [ ]:
# Extract data from each record set, referenced by their @id
dataframes = dict()

# Gather all record set @id values
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    # Load records for each record set by @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns for record set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"\nNo records found for record set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping by key attributes for further analysis.

_Note: Please edit the field `numeric_field_id` and `group_field_id` to match those available in your selected record set._

Below, we demonstrate on the first record set found with numerical data.

In [ ]:
# Pick first record set that has data and at least one numeric column
import numpy as np

record_set_id = None
numeric_field_id = None
group_field_id = None

# Auto-detect an appropriate numeric field for this EDA example
for rs in record_sets:
    df = dataframes.get(rs.id)
    if df is not None and not df.empty:
        # Find first numeric column from field definitions
        for field in rs.fields:
            # We'll use dataType to guess numeric, e.g., 'Float', 'Integer', 'Number'
            if field.data_type in ('Float', 'Integer', 'Number') and field.id in df.columns:
                record_set_id = rs.id
                numeric_field_id = field.id
                break
        # Find first non-numeric (categorical) field for grouping
        for field in rs.fields:
            if (field.data_type in ('Text', 'String') or 'desc' in field.name.lower()) and field.id in df.columns and field.id!=numeric_field_id:
                group_field_id = field.id
                break
        if record_set_id and numeric_field_id:
            break

if record_set_id and numeric_field_id:
    df = dataframes[record_set_id]
    # Convert to numeric, coerce errors
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Filtering: keep where value > median
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records from record set {record_set_id} where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalizing
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping if possible
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by group {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric data found for EDA demonstration.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the numeric field and its mean per group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.show()

    # Plot mean per group if group field available
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.barplot(y=grouped_df[group_field_id], x=grouped_df[numeric_field_id], orient='h')
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No visualization possible without suitable numeric data.")

## 6. Conclusion
In this notebook, we demonstrated loading and basic exploration of the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the `mlcroissant` Python package. 

- The Croissant schema allows structured access to record sets and fields by their `@id`.
- We reviewed available record sets and fields, loaded data, filtered and normalized numeric values, and visualized basic distributions.

Refer to the dataset documentation for details of the schema and each field. You can customize the exploratory steps according to research or analytics needs.